In [53]:
from sage.all import *
import string
pretty_print_default(True)


In [101]:
x, t = var('x t')

L = lie_algebras.sl(QQ, 2, representation="matrix")

Ep, Em, H = L.gens()
Ep = Ep.matrix()
Em = Em.matrix()
H = H.matrix()

v = function('v')(x, t)
A0 = v*H

# R = LaurentPolynomialRing(QQ, 'lam')
# lam = R.gen()
lam = var('lam')

E = Ep + lam*Em

In [88]:
n = int(input())
fields = {}
grade = 0
count = 0

for y in range(0, int(n+1+(n+1)/2)):
    print(y % 2)
    letter = string.ascii_lowercase[y]
    exec(f"{letter} = function('{letter}')(x, t)")

    if grade not in fields:
        fields[grade] = []

    fields[grade].append(eval(letter))
    
    count += 1

    if grade % 2 == 0 and count == 1:
        grade += 1
        count = 0
    
    elif grade % 2 == 1 and count == 2:
        grade += 1
        count = 0
    
fields

0
1
0
1
0
1


{0: [a(x, t)], 1: [b(x, t), c(x, t)], 2: [d(x, t)], 3: [e(x, t), f(x, t)]}

In [90]:
def comm(A,B):
    return A*B - B*A

In [102]:
grades = {}
fields[n][0] = 1
fields[n][1] = 1

for g in range(n, -1, -1):
    this_grade = zero_matrix(2, 2) 

    if g != 0:
        if g % 2 != 0:
            D_n = lam**((g-1)//2)*(fields[g][0]*Ep + fields[g][1]*lam*Em)
        else: 
            D_n = fields[g][0]*(lam**(g//2))*H 
        
        if (g-1) % 2 != 0:
            D_n_minus_one = lam**((g-1)//2)*(fields[g-1][0]*Ep + fields[g-1][1]*lam*Em)
        else: 
            D_n_minus_one = fields[g-1][0]*(lam**((g-1)//2))*H 
        this_grade += diff(D_n, x) 
        this_grade += comm(E, D_n_minus_one) + comm(A0, D_n) 

    else:
        this_grade -= diff(A0, t)
        D_n = fields[g][0]*(lam**(g//2))*H
        this_grade += diff(D_n, x) + comm(A0, D_n)
    grades[g] = this_grade
    # diff(D_n, x)        
grades
    


{3: [                                0    -2*lam*d(x, t) + 2*lam*v(x, t)]
 [2*lam^2*d(x, t) - 2*lam^2*v(x, t)                                 0],
 2: [-lam*b(x, t) + lam*c(x, t) + lam*diff(d(x, t), x)                                                 0]
 [                                                0  lam*b(x, t) - lam*c(x, t) - lam*diff(d(x, t), x)],
 1: [                                                            0              2*b(x, t)*v(x, t) - 2*a(x, t) + diff(b(x, t), x)]
 [-2*lam*c(x, t)*v(x, t) + 2*lam*a(x, t) + lam*diff(c(x, t), x)                                                             0],
 0: [ diff(a(x, t), x) - diff(v(x, t), t)                                    0]
 [                                   0 -diff(a(x, t), x) + diff(v(x, t), t)]}

In [ ]:
def split_lambda(expr):
    expr = expand(expr)
    powers = expr.collect(lam).coefficients(lam)
    out = {}
    for coeff, power in powers:
        out[power] = expand(coeff)
    return out

In [ ]:

for gr in grades:
